# 05 - Data Quality Metrics (Gold)

Adds **one row per checked table per run** to `gold.data_quality_metrics` (the dashboard shows the latest run).
`quality_status` is `FAIL` when any issue count is greater than zero.

| Column | Meaning |
|---|---|
| `duplicate_count` | rows that repeat the table's key |
| `null_count` | NULLs in province, crop, crop_group, year, quarter, production |
| `invalid_province_count` / `invalid_crop_count` | present but blank |
| `invalid_crop_group_count` | not PALAY / CORN / OTHER |
| `invalid_quarter_count` | not Q1 - Q4 |

In [0]:
import uuid

from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType, StructField, StructType

from common.delta_io import write_delta_table
from configs.etl_config import (
    SILVER_CROP_WEATHER_TABLE,
    GOLD_PROVINCE_SUMMARY_TABLE,
    GOLD_CROP_GROUP_TRENDS_TABLE,
    GOLD_CROP_PERFORMANCE_TABLE,
    GOLD_FORECAST_FEATURES_TABLE,
    GOLD_QUALITY_METRICS_TABLE,
    CROP_KEY_COLUMNS,
    VALID_QUARTERS,
    VALID_CROP_GROUPS,
)

run_id = str(uuid.uuid4())

# table to check -> columns that must be unique together
tables_to_check = {
    SILVER_CROP_WEATHER_TABLE: CROP_KEY_COLUMNS,
    GOLD_PROVINCE_SUMMARY_TABLE: ["province", "year", "quarter"],
    GOLD_CROP_GROUP_TRENDS_TABLE: ["year", "quarter", "crop_group"],
    GOLD_CROP_PERFORMANCE_TABLE: CROP_KEY_COLUMNS,
    GOLD_FORECAST_FEATURES_TABLE: CROP_KEY_COLUMNS,
}

# COMMAND ----------

def count_rows_where(condition):
    """Aggregate expression: number of rows where `condition` is true."""
    return F.sum(F.when(condition, 1).otherwise(0))


def check_table(table_name: str, key_columns: list) -> tuple:
    """Return one row of quality counts for a table. Checks are skipped when a column is missing."""
    df = spark.table(table_name)
    columns = df.columns

    null_checks = [
        count_rows_where(F.col(c).isNull())
        for c in ["province", "crop", "crop_group", "year", "quarter", "production"]
        if c in columns
    ]
    counts = df.agg(
        F.count(F.lit(1)).alias("row_count"),
        sum(null_checks, F.lit(0)).alias("null_count"),
        (count_rows_where(F.trim(F.col("province")) == "") if "province" in columns else F.lit(0)).alias("invalid_province"),
        (count_rows_where(F.trim(F.col("crop")) == "") if "crop" in columns else F.lit(0)).alias("invalid_crop"),
        (count_rows_where(~F.col("crop_group").isin(VALID_CROP_GROUPS)) if "crop_group" in columns else F.lit(0)).alias("invalid_crop_group"),
        (count_rows_where(~F.col("quarter").isin(VALID_QUARTERS)) if "quarter" in columns else F.lit(0)).alias("invalid_quarter"),
    ).first()

    row_count = counts["row_count"]
    duplicate_count = row_count - df.select(*key_columns).distinct().count()

    issue_counts = [
        duplicate_count, counts["null_count"], counts["invalid_province"],
        counts["invalid_crop"], counts["invalid_crop_group"], counts["invalid_quarter"],
    ]
    issue_counts = [int(value or 0) for value in issue_counts]
    status = "FAIL" if any(value > 0 for value in issue_counts) else "PASS"

    return (run_id, table_name, int(row_count), *issue_counts, status)


# COMMAND ----------

metrics_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("row_count", LongType(), False),
    StructField("duplicate_count", LongType(), False),
    StructField("null_count", LongType(), False),
    StructField("invalid_province_count", LongType(), False),
    StructField("invalid_crop_count", LongType(), False),
    StructField("invalid_crop_group_count", LongType(), False),
    StructField("invalid_quarter_count", LongType(), False),
    StructField("quality_status", StringType(), False),
])

metrics_rows = [check_table(table, keys) for table, keys in tables_to_check.items()]

metrics_df = (
    spark.createDataFrame(metrics_rows, metrics_schema)
    .withColumn("run_timestamp", F.current_timestamp())
)

write_delta_table(metrics_df, GOLD_QUALITY_METRICS_TABLE, mode="append")
metrics_df.show(truncate=False)